In [1]:
from functools import lru_cache

import numpy as np
import panel as pn
import plotly.graph_objects as go

pn.extension('plotly')



# Constantes geométricas
TWO_PI = 2.0 * np.pi
SQRT2 = np.sqrt(2.0)
DECAY_RATE = 2.0  # autovalor no nulo de L


def solve_laplacian(x1_0_deg: float, x2_0_deg: float, T: float, n_samples: int = 400) -> dict:
    """Solución analítica exacta del flujo laplaciano en 2 nodos.

    Modo consenso (autovalor 0): s(t) = (x1+x2)/2, constante.
    Modo disenso (autovalor 2): d(t) = (x1-x2)/2, decae como exp(-2t).
    """
    x1_0 = np.deg2rad(x1_0_deg)
    x2_0 = np.deg2rad(x2_0_deg)

    s0 = 0.5 * (x1_0 + x2_0)           # modo consenso
    d0 = 0.5 * (x1_0 - x2_0)           # modo disenso

    t = np.linspace(0.0, T, n_samples)
    decay = np.exp(-DECAY_RATE * t)
    d_t = d0 * decay

    x1 = s0 + d_t
    x2 = s0 - d_t
    x = np.column_stack([x1, x2])

    delta = x2 - x1
    return {
        't': t,
        'x_rad': x,
        'x_deg_unwrapped': np.rad2deg(x),
        'x_deg_mod': np.rad2deg(x % TWO_PI),
        'delta_rad': delta,
        'delta_deg': np.rad2deg(delta),
        'dist_diag': np.abs(delta) / SQRT2,
        'consensus': s0,
        'disagreement_0': d0,
        'disagreement_t': -d_t,  # proyección ortogonal a D: d(t)*(1,-1)/sqrt(2) tiene norma |d_t|
    }


def torus_embedding(theta1, theta2, R=3.0, r=1.0):
    """Embebe S^1 x S^1 en R^3 como toro de revolución."""
    cos_t2 = np.cos(theta2)
    x = (R + r * cos_t2) * np.cos(theta1)
    y = (R + r * cos_t2) * np.sin(theta1)
    z = r * np.sin(theta2)
    return x, y, z


@lru_cache(maxsize=16)
def make_torus_mesh(R: float = 3.0, r: float = 1.0, nu: int = 70, nv: int = 40):
    """Malla del toro cacheada por (R, r, nu, nv)."""
    u = np.linspace(0, TWO_PI, nu)
    v = np.linspace(0, TWO_PI, nv)
    U, V = np.meshgrid(u, v)
    cos_V = np.cos(V)
    X = (R + r * cos_V) * np.cos(U)
    Y = (R + r * cos_V) * np.sin(U)
    Z = r * np.sin(V)
    return X, Y, Z


@lru_cache(maxsize=64)
def _solve_cached(x1_0_deg: float, x2_0_deg: float, T: float, n_samples: int):
    return solve_laplacian(x1_0_deg, x2_0_deg, T, n_samples)


def get_solution(x1_0_deg, x2_0_deg, T, n_samples):
    # Redondeamos para que cambios infinitesimales no invaliden la caché
    return _solve_cached(round(x1_0_deg, 2), round(x2_0_deg, 2), round(T, 2), int(n_samples))



# Paleta consistente para toda la app
COLOR_TRAJ = '#2563eb'       # azul trayectoria
COLOR_NOW = '#dc2626'        # rojo estado actual
COLOR_DIAG = '#059669'       # verde diagonal
COLOR_X1 = '#2563eb'
COLOR_X2 = '#ea580c'
COLOR_DECAY_TH = '#6b7280'   # gris teórico

BASE_LAYOUT = dict(
    margin=dict(l=40, r=40, t=50, b=40),
    template='plotly_white',
)


def make_torus_figure(data, idx, trail, R_major, r_minor):
    xmod = data['x_deg_mod']
    th1 = np.deg2rad(xmod[:, 0])
    th2 = np.deg2rad(xmod[:, 1])
    Xp, Yp, Zp = torus_embedding(th1, th2, R=R_major, r=r_minor)

    start = max(0, idx - trail)
    X, Y, Z = make_torus_mesh(R=R_major, r=r_minor)

    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z, opacity=0.22, showscale=False,
        colorscale=[[0, '#e0e7ff'], [1, '#c7d2fe']], name='Toro',
    ))
    # Curva diagonal theta1 = theta2 proyectada al toro (para referencia)
    th_diag = np.linspace(0, TWO_PI, 200)
    Xd, Yd, Zd = torus_embedding(th_diag, th_diag, R=R_major, r=r_minor)
    fig.add_trace(go.Scatter3d(
        x=Xd, y=Yd, z=Zd, mode='lines', name='Diagonal D (toro)',
        line=dict(width=3, color=COLOR_DIAG, dash='dash'), opacity=0.6,
    ))
    fig.add_trace(go.Scatter3d(
        x=Xp[start:idx + 1], y=Yp[start:idx + 1], z=Zp[start:idx + 1],
        mode='lines', name='Trayectoria', line=dict(width=6, color=COLOR_TRAJ),
    ))
    fig.add_trace(go.Scatter3d(
        x=[Xp[idx]], y=[Yp[idx]], z=[Zp[idx]],
        mode='markers', name='Estado actual',
        marker=dict(size=7, color=COLOR_NOW),
    ))
    fig.update_layout(
        title=f'Flujo en el toro · t = {data["t"][idx]:.2f}',
        height=520,
        margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(aspectmode='data',
                   xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    )
    return fig


def make_angles_figure(data, idx):
    t = data['t']
    xdeg = data['x_deg_unwrapped']
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=xdeg[:, 0], mode='lines',
                             name='x₁(t)', line=dict(color=COLOR_X1, width=2)))
    fig.add_trace(go.Scatter(x=t, y=xdeg[:, 1], mode='lines',
                             name='x₂(t)', line=dict(color=COLOR_X2, width=2)))
    # Línea vertical indicando el tiempo actual
    fig.add_vline(x=t[idx], line=dict(color=COLOR_NOW, width=1, dash='dot'))
    fig.update_layout(
        title='Coordenadas desenrolladas en ℝ²',
        xaxis_title='t', yaxis_title='grados',
        height=260, legend=dict(orientation='h', y=1.15),
        **BASE_LAYOUT,
    )
    return fig


def make_diag_figure(data, idx):
    t = data['t']
    dist = data['dist_diag']
    # Curva teórica de decaimiento: dist(t) = dist(0) * exp(-2t)
    dist_theory = dist[0] * np.exp(-DECAY_RATE * t)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=dist, mode='lines',
                             name='dist(x(t), D) medida',
                             line=dict(color=COLOR_DIAG, width=3)))
    fig.add_trace(go.Scatter(x=t, y=dist_theory, mode='lines',
                             name='‖x₀-x̄‖·e^(−2t)',
                             line=dict(color=COLOR_DECAY_TH, width=1.5, dash='dash')))
    fig.add_vline(x=t[idx], line=dict(color=COLOR_NOW, width=1, dash='dot'))
    fig.update_layout(
        title='Distancia a la diagonal vs. decaimiento teórico e^(−λ₁ t)',
        xaxis_title='t', yaxis_title='dist(x, D) [rad]',
        height=260, legend=dict(orientation='h', y=1.15),
        **BASE_LAYOUT,
    )
    return fig


def make_plane_figure(data, idx):
    xdeg = data['x_deg_unwrapped']
    x1, x2 = xdeg[:, 0], xdeg[:, 1]

    # Rango para dibujar la diagonal
    lo = min(x1.min(), x2.min()) - 20
    hi = max(x1.max(), x2.max()) + 20

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines',
                             name='Diagonal D',
                             line=dict(color=COLOR_DIAG, dash='dash', width=2)))
    fig.add_trace(go.Scatter(x=x1, y=x2, mode='lines',
                             name='Trayectoria',
                             line=dict(color=COLOR_TRAJ, width=2)))
    fig.add_trace(go.Scatter(x=[x1[0]], y=[x2[0]], mode='markers',
                             name='x(0)', marker=dict(size=10, color=COLOR_X2, symbol='circle-open')))
    fig.add_trace(go.Scatter(x=[x1[idx]], y=[x2[idx]], mode='markers',
                             name='x(t)', marker=dict(size=12, color=COLOR_NOW)))
    fig.update_layout(
        title='Espacio de estados ℝ² · trayectoria cayendo a D',
        xaxis_title='x₁ [grados]', yaxis_title='x₂ [grados]',
        height=380, legend=dict(orientation='h', y=1.12),
        yaxis=dict(scaleanchor='x', scaleratio=1),
        **BASE_LAYOUT,
    )
    return fig


def make_modes_figure(data, idx):
    """Descomposición espectral: modo consenso (constante) vs disenso (decae)."""
    t = data['t']
    consensus_deg = np.rad2deg(np.full_like(t, data['consensus']))
    disagree_deg = np.rad2deg(data['disagreement_0']) * np.exp(-DECAY_RATE * t)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=consensus_deg, mode='lines',
                             name='Modo consenso (λ₀=0)',
                             line=dict(color='#7c3aed', width=2)))
    fig.add_trace(go.Scatter(x=t, y=disagree_deg, mode='lines',
                             name='Modo disenso (λ₁=2)',
                             line=dict(color='#f59e0b', width=2)))
    fig.add_hline(y=0, line=dict(color='#9ca3af', width=1))
    fig.add_vline(x=t[idx], line=dict(color=COLOR_NOW, width=1, dash='dot'))
    fig.update_layout(
        title='Descomposición en autovectores de L',
        xaxis_title='t', yaxis_title='amplitud [grados]',
        height=260, legend=dict(orientation='h', y=1.15),
        **BASE_LAYOUT,
    )
    return fig

# --- Widgets ---
x1_w = pn.widgets.FloatSlider(name='x₁(0) [grados]', start=0, end=360, step=1, value=40)
x2_w = pn.widgets.FloatSlider(name='x₂(0) [grados]', start=0, end=360, step=1, value=260)
T_w = pn.widgets.FloatSlider(name='Tiempo total T', start=0.5, end=10.0, step=0.1, value=3.0)
n_w = pn.widgets.IntSlider(name='Muestras temporales', start=100, end=1000, step=50, value=400)
trail_w = pn.widgets.IntSlider(name='Cola visible (toro)', start=5, end=400, step=5, value=120)
R_w = pn.widgets.FloatSlider(name='Radio mayor del toro', start=1.5, end=5.0, step=0.1, value=3.0)
r_w = pn.widgets.FloatSlider(name='Radio menor del toro', start=0.3, end=2.0, step=0.05, value=1.0)
player = pn.widgets.Player(name='t', start=0, end=399, step=1, interval=60, value=0, loop_policy='loop')
reset_btn = pn.widgets.Button(name='Restablecer', button_type='primary')

DEFAULTS = dict(x1=40, x2=260, T=3.0, n=400, trail=120, R=3.0, r=1.0)


def _reset(event=None):
    x1_w.value = DEFAULTS['x1']
    x2_w.value = DEFAULTS['x2']
    T_w.value = DEFAULTS['T']
    n_w.value = DEFAULTS['n']
    trail_w.value = DEFAULTS['trail']
    R_w.value = DEFAULTS['R']
    r_w.value = DEFAULTS['r']
    player.value = 0


reset_btn.on_click(_reset)

# Sincronizar el rango del player con n_samples sin efectos colaterales dentro de callbacks
def _sync_player(event):
    player.end = max(0, event.new - 1)
    if player.value > player.end:
        player.value = player.end


n_w.param.watch(_sync_player, 'value')


# --- Panel de estado (texto) ---
@pn.depends(x1_w, x2_w, T_w, n_w, player)
def status_panel(x1_0, x2_0, T, n, time_index):
    data = get_solution(x1_0, x2_0, T, n)
    idx = min(time_index, len(data['t']) - 1)
    t = data['t'][idx]
    xmod = data['x_deg_mod'][idx]
    delta = data['delta_deg'][idx]
    dist = data['dist_diag'][idx]
    dist_0 = data['dist_diag'][0]
    # Tasa medida (si hay decaimiento apreciable)
    if dist_0 > 1e-9 and dist > 1e-12:
        rate_measured = -np.log(dist / dist_0) / t if t > 1e-6 else np.nan
    else:
        rate_measured = np.nan

    return pn.pane.Markdown(f"""
### Estado en t = {t:.3f}

| Cantidad | Valor |
|---|---|
| x₁(t) mod 360 | {xmod[0]:7.2f}° |
| x₂(t) mod 360 | {xmod[1]:7.2f}° |
| x₂ − x₁ | {delta:7.2f}° |
| dist(x, D) | {dist:.6f} rad |
| Tasa medida | {rate_measured:.3f} |
| Tasa teórica λ₁ | {DECAY_RATE:.3f} |

**Espectro de L:** λ₀=0, λ₁=2. La convergencia es exponencial con tasa λ₁.
""", sizing_mode='stretch_width')


# --- Paneles gráficos ---
# Cada uno depende SOLO de los widgets que realmente necesita → menos re-renderizados

@pn.depends(x1_w, x2_w, T_w, n_w, trail_w, R_w, r_w, player)
def torus_panel(x1_0, x2_0, T, n, trail, R, r, k):
    data = get_solution(x1_0, x2_0, T, n)
    idx = min(k, len(data['t']) - 1)
    return pn.pane.Plotly(make_torus_figure(data, idx, trail, R, r),
                          config={'responsive': True})


@pn.depends(x1_w, x2_w, T_w, n_w, player)
def angles_panel(x1_0, x2_0, T, n, k):
    data = get_solution(x1_0, x2_0, T, n)
    idx = min(k, len(data['t']) - 1)
    return pn.pane.Plotly(make_angles_figure(data, idx),
                          config={'responsive': True})


@pn.depends(x1_w, x2_w, T_w, n_w, player)
def modes_panel(x1_0, x2_0, T, n, k):
    data = get_solution(x1_0, x2_0, T, n)
    idx = min(k, len(data['t']) - 1)
    return pn.pane.Plotly(make_modes_figure(data, idx),
                          config={'responsive': True})


@pn.depends(x1_w, x2_w, T_w, n_w, player)
def diag_panel(x1_0, x2_0, T, n, k):
    data = get_solution(x1_0, x2_0, T, n)
    idx = min(k, len(data['t']) - 1)
    return pn.pane.Plotly(make_diag_figure(data, idx),
                          config={'responsive': True})


@pn.depends(x1_w, x2_w, T_w, n_w, player)
def plane_panel(x1_0, x2_0, T, n, k):
    data = get_solution(x1_0, x2_0, T, n)
    idx = min(k, len(data['t']) - 1)
    return pn.pane.Plotly(make_plane_figure(data, idx),
                          config={'responsive': True})



controls = pn.Card(
    pn.pane.Markdown('**Condiciones iniciales**'),
    x1_w, x2_w,
    pn.pane.Markdown('**Simulación**'),
    T_w, n_w,
    pn.pane.Markdown('**Visualización**'),
    trail_w, R_w, r_w,
    pn.pane.Markdown('**Animación**'),
    player, reset_btn,
    title='Controles', width=340, collapsed=False,
)

header = pn.pane.Markdown("""
## Flujo laplaciano en el toro y atracción hacia la diagonal

Dos variables acopladas por $\\dot x = -Lx$. En $\\mathbb R^2$ la diagonal $D = \\{x_1 = x_2\\}$ es atractora con tasa $\\lambda_1 = 2$ (segundo autovalor de $L$). Proyectando mod $2\\pi$ visualizamos la trayectoria en el toro $T^2 = S^1 \\times S^1$.
""")

right_column = pn.Column(
    status_panel,
    pn.Tabs(
        ('Toro', torus_panel),
        ('Plano ℝ²', plane_panel),
        ('Coordenadas x₁, x₂', angles_panel),
        ('Modos espectrales', modes_panel),
        ('Convergencia a D', diag_panel),
    ),
)

app = pn.Column(header, pn.Row(controls, right_column))
app

Column
    [0] Markdown(str)
    [1] Row
        [0] Card(title='Controles', width=340)
            [0] Markdown(str)
            [1] FloatSlider(end=360, name='x₁(0) [grados]', step=1, value=40)
            [2] FloatSlider(end=360, name='x₂(0) [grados]', step=1, value=260)
            [3] Markdown(str)
            [4] FloatSlider(end=10.0, name='Tiempo total T', start=0.5, value=3.0)
            [5] IntSlider(end=1000, name='Muestras temporales', start=100, step=50, value=400)
            [6] Markdown(str)
            [7] IntSlider(end=400, name='Cola visible (toro)', start=5, step=5, value=120)
            [8] FloatSlider(end=5.0, name='Radio mayor del toro', start=1.5, value=3.0)
            [9] FloatSlider(end=2.0, name='Radio menor del toro', start=0.3, step=0.05, value=1.0)
            [10] Markdown(str)
            [11] Player(end=399, interval=60, loop_policy='loop', name='t')
            [12] Button(button_type='primary', name='Restablecer')
        [1] Column
            [0] ParamFunction(function, _pane=Markdown, defer_load=False)
            [1] Tabs
                [0] ParamFunction(function, _pane=Plotly, defer_load=False, name='Toro')
                [1] ParamFunction(function, _pane=Plotly, defer_load=False, name='Plano ℝ²')
                [2] ParamFunction(function, _pane=Plotly, defer_load=False, name='Coordenadas x₁, x₂')
                [3] ParamFunction(function, _pane=Plotly, defer_load=False, name='Modos espectrales')
                [4] ParamFunction(function, _pane=Plotly, defer_load=False, name='Convergencia a D')